In [2]:
import os
import pandas as pd

# Rutas del proyecto
PATH_RAIZ = r"C:\Users\bootr\Documents\proyectos\PROYECTO ML\especuladores"
DIR_DATA = os.path.join(PATH_RAIZ, "data")

PATH_INPUT_FUENTES = os.path.join(DIR_DATA, "raw", "df_fuentes_unidas.csv")
PATH_OUTPUT_PROCESSED = os.path.join(DIR_DATA, "processed", "df_processed.csv")

print(f"📡 Leyendo datos unificados desde: {PATH_INPUT_FUENTES}")
df_eda = pd.read_csv(PATH_INPUT_FUENTES)
print(f"✅ Dataset cargado correctamente. Registros a procesar: {len(df_eda)}")

📡 Leyendo datos unificados desde: C:\Users\bootr\Documents\proyectos\PROYECTO ML\especuladores\data\raw\df_fuentes_unidas.csv
✅ Dataset cargado correctamente. Registros a procesar: 1250


In [3]:
import re

print("🔬 Ejecutando limpieza y auditoría sobre el campo de licencias turísticas...")

def auditar_campo_licencia(texto):
    if not isinstance(texto, str) or texto.strip() == "" or texto.upper() == 'SIN_REGISTRO':
        return "ILEGAL_SIN_REGISTRO"
    cadena = texto.strip().upper()
    
    # Captura el patrón del Catastro largo metido en la licencia
    if "ESFCTU" in cadena or len(cadena) > 30:
        match = re.search(r'(EBI\d+|ESS\d+|LSS\d+)', cadena)
        return f"CORREGIDA_{match.group(1)}" if match else "FRAUDE_FORMATO_CATASTRO"
        
    if cadena == "EXENTO":
        return "EXENTO_OFICIAL"
    return f"OK_{cadena}"

# Aplicamos la función limpiadora
df_eda['estado_licencia_auditada'] = df_eda['license'].apply(auditar_campo_licencia)

# Creamos la etiqueta de fraude (0 o 1) que usará la IA más adelante
df_eda['y_fraude_administrativo'] = df_eda['estado_licencia_auditada'].str.contains('FRAUDE|ILEGAL').astype(int)

# Eliminamos duplicados por ID de piso
df_eda = df_eda.drop_duplicates(subset=['id'])
df_eda['price_clean'] = df_eda['price_clean'].fillna(df_eda['price_clean'].median())
df_eda['availability_365'] = df_eda['availability_365'].fillna(180)

print(f"✅ Limpieza completada. Filas listas: {len(df_eda)}")

🔬 Ejecutando limpieza y auditoría sobre el campo de licencias turísticas...
✅ Limpieza completada. Filas listas: 1250
